# SAGE-CP v4.1: NHANES external validation with design-based intervals

Same code as v4 (all studies remain available by editing `STUDIES` in Section 1); this configuration re-runs only the NHANES external validation and adds 1,000 Rao–Wu replicates over NHANES PSUs within strata, so every NHANES quantity (including coverage of undiagnosed diabetes) gets a design-based interval.

**Inputs:** `NEWDATA` (raw BRFSS `LLCP_2022.XPT`) and `nhanes-2021-2023` (six NHANES 2021–2023 files). Internet off. Run All; download `SAGE_CP_v4_results.zip`.


## 1. Configuration

In [ ]:
# ======================= 1. Configuration (SAGE-CP v4 revision studies) =======================
# v4 adds, on top of v3 (all v3 functions are reused verbatim):
#   RAWP       raw BRFSS 2022 as the PRIMARY analysis: leave-one-state-out, logistic regression, 3 seeds, design-weighted
#              metrics, tuned Gibbs baseline (LR, 2,000 calibration points per label), risk-cutoff comparison, and a
#              design-based (Rao-Wu rescaled bootstrap over PSUs within strata) standard error for every pooled contrast
#   RAWP_MISIND  same, numeric missing values encoded with missing indicators (sensitivity)
#   RAWP_MI      same, multiple imputation (m = 5, chained equations, outcome not used) (sensitivity)
#   RAWP_CB      same, CatBoost with isotonic calibration (deferral sensitivity)
#   RAWP_FULL    same, logistic regression on the full raw feature set (deferral sensitivity)
#   LOSO_G     Kaggle leave-one-state-out with the tuned Gibbs baseline (LR, 2,000 calibration points per label)
#   WEST       concept-shift diagnostics by held-out region (Kaggle extract, v2.1 full feature set, LR and LightGBM)
#   NHANES     external validation with MEASURED diabetes (HbA1c >= 6.5%) in NHANES August 2021-August 2023:
#              N1 = calibrate on BRFSS 2022, evaluate in NHANES; N2 = the same pipeline within NHANES
# Four studies, each run unit by unit and saved after every unit, so the notebook can be stopped and RESUMED
# (re-running skips units already in results_v3/units.csv):
#   R5   five held-out regions on the Kaggle extract (the paper's design) + the tuned Gibbs-Cherian-Candes baseline
#   LOSO leave-one-state-out on the Kaggle extract (about 54 held-out units instead of 5)
#   RAW  leave-one-state-out on the raw CDC BRFSS 2022 file: all respondents (no complete-case filter), metrics also
#        weighted by the BRFSS design weight _LLCPWT
#   TEMP temporal + geographic: models and calibration on raw 2022, evaluation on raw 2023 (weighted), per state
# Every decision rule and setting is fixed here, before any result is seen.
import os, glob, json, time, math, hashlib, platform, sys, io, zipfile, urllib.request, warnings, traceback
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

CFG3 = dict(
    VERSION="SAGE-CP v4 revision studies (RAWP, LOSO_G, WEST, NHANES)",
    STUDIES=["NHANES"],                             # v4.1: NHANES only, with design-based replicates (full v4 list: NHANES, WEST, LOSO_G, RAWP, RAWP_MISIND, RAWP_MI, RAWP_FULL, RAWP_CB)
    NHANES_BOOT=True,
    ALPHA=0.10,
    OUTCOMES=["HadHeartAttack", "HadStroke", "HadDiabetes"],
    MODELS={"RAWP": ["LR"], "RAWP_MISIND": ["LR"], "RAWP_MI": ["LR"], "RAWP_FULL": ["LR"], "RAWP_CB": ["CBCAL"], "LOSO_G": ["LR"],
            "WEST": ["LR", "LGBM"], "NHANES": ["LR", "LGBM"]},
    SEEDS={"RAWP": [0, 1, 2], "RAWP_MISIND": [0], "RAWP_MI": [0], "RAWP_FULL": [0], "RAWP_CB": [0], "LOSO_G": [0],
           "WEST": [0, 1, 2, 3, 4], "NHANES": list(range(10))},
    MI_M=5, MI_FIT_ROWS=20000, MI_MAX_ITER=5,           # multiple imputation: imputations, rows used to fit the imputer, iterations
    BOOT_DESIGN=1000, BOOT_DESIGN_SEED=4242,           # Rao-Wu replicates (RAWP seed 0 and NHANES)
    NHANES_FILES=["DEMO_L", "DIQ_L", "GHB_L", "GLU_L", "BMX_L", "SMQ_L"],
    NHANES_URL="https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/{}.xpt",
    HBA1C_CUT=6.5, FPG_CUT=126.0,
    # PRIMARY feature set = no_proxy (the 13 possible post-diagnosis proxies removed; v2.1 features_no_proxy.csv).
    # LOSO_FULL repeats leave-one-state-out with the full v2.1 set (Kaggle only) as a sensitivity analysis.
    FEATURES_NO_PROXY=["Sex", "MentalHealthDays", "PhysicalActivities", "SleepHours", "DeafOrHardOfHearing",
                       "BlindOrVisionDifficulty", "DifficultyConcentrating", "SmokerStatus", "ECigaretteUsage",
                       "RaceEthnicityCategory", "AgeCategory", "HeightInMeters", "WeightInKilograms", "BMI",
                       "AlcoholDrinkers", "CovidPos"],
    EXTRA_ALPHAS=[0.20],                              # informativeness at alpha = 0.20 (ClassCond, GroupCond, SAGE-CP)
    N_CAL=20000, N_SRC_TEST=20000, N_DOM=20000, MAX_TRAIN=60000,   # as in v2.1
    CLIP_Q=0.99, DOMAIN_C=0.1, MIN_ESS=1,
    MIN_GROUP=100, MIN_POS=30,                        # eligibility of a group for worst-group metrics (as in v2.1)
    MIN_STATE_ROWS=500,                               # a state with fewer rows is not a held-out unit (rows stay in the source)
    REF_SIMS=2000,                                    # simulations of the exact worst-group reference (Proposition 3)
    # ---- tuned Gibbs, Cherian & Candes (2025) conditional-calibration baseline ----
    GIBBS=dict(studies=["RAWP", "LOSO_G"],          # LR only (scorecard item 7)
               score_grid=1e-3,                       # scores rounded UP to this grid (conservative) to compress the LP
               n_cal_max=2000,                        # calibration points per label used by the LP (random subsample)
               wbins=[0, 4],                          # candidate numbers of density-ratio bins in the function class
               race=[False, True],                    # candidate: include race indicators
               # selection rule (label-free for the target): for each candidate class, fit the unaugmented quantile
               # regression (level (1-alpha)(n+1)/n), build sets on s_test and estimate every eligible calibrated-group
               # positive-class coverage with the unclipped importance weights; pick the class with the smallest estimated
               # worst-group shortfall below 1-alpha, ties (within tie_tol) broken by the smaller estimated set size.
               # The selected class is then refitted with the exact augmented procedure for the evaluation data.
               tie_tol=0.005),
    BOOT_STATES=5000, BOOT_SEED=2026,                 # bootstrap over held-out units (LOSO, RAW, TEMP)
    TIME_BUDGET_MIN=660,                              # stop starting new units after this; re-run the notebook to resume
    RAW_URLS={2022: "https://www.cdc.gov/brfss/annual_data/2022/files/LLCP2022XPT.zip",
              2023: "https://www.cdc.gov/brfss/annual_data/2023/files/LLCP2023XPT.zip"},
    QUICK=os.environ.get("QUICK", "0") == "1",
)
if CFG3["QUICK"]:   # offline synthetic test only
    CFG3.update(SEEDS={k: [0] for k in CFG3["SEEDS"]}, N_CAL=1500, N_SRC_TEST=1500, N_DOM=1500, MAX_TRAIN=3000,
                MIN_GROUP=30, MIN_POS=8, MIN_STATE_ROWS=200, REF_SIMS=200, BOOT_STATES=300)
    CFG3["GIBBS"].update(n_cal_max=800, wbins=[0, 3])
    CFG3.update(SEEDS={k: [0] for k in CFG3["SEEDS"]}, MI_M=2, MI_FIT_ROWS=2000, BOOT_DESIGN=20)
ALPHA = CFG3["ALPHA"]
OUT = "/kaggle/working/results_v4" if os.path.isdir("/kaggle/working") else os.environ.get("SAGE_OUT", "./results_v4")
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
# resume across sessions: copy finished units from a previous run attached as an input (any folder named results_v3)
import shutil
for f in glob.glob("/kaggle/input/**/results_v4/units/*.csv", recursive=True) + glob.glob("/kaggle/input/**/results_v4/done_units.json", recursive=True):
    dst = f"{OUT}/units/{os.path.basename(f)}" if "/units/" in f else f"{OUT}/{os.path.basename(f)}"
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst): shutil.copy(f, dst)
CFG3_HASH = hashlib.sha256(json.dumps(CFG3, sort_keys=True, default=str).encode()).hexdigest()[:12]
HW = dict(cpu_count=os.cpu_count(), ram_gb=round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1) if hasattr(os, "sysconf") else None,
          processor=platform.processor() or platform.machine())
print("hardware:", HW)
json.dump(dict(CFG3=CFG3, cfg_hash=CFG3_HASH, python=sys.version, platform=platform.platform(), hardware=HW),
          open(f"{OUT}/config_v4.json", "w"), indent=2, default=str)
CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append(dict(check=name, passed=bool(ok), detail=str(detail)))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + ("" if ok else f" | {detail}"))
    return bool(ok)
print(CFG3["VERSION"], "| config hash", CFG3_HASH, "| output", OUT)


## 2. Data

In [ ]:
# ======================= 2. Data: Kaggle extract and raw CDC BRFSS files, mapped to one standard frame =======================
# Standard frame: state (name), Sex, Age (18-44 / 45-64 / 65+), Race, the three outcomes coded 0/1/NaN, wdesign (BRFSS
# design weight; 1 for the Kaggle extract) and the model features. Outcomes and other diagnoses never enter a feature set.
FIPS = {1: "Alabama", 2: "Alaska", 4: "Arizona", 5: "Arkansas", 6: "California", 8: "Colorado", 9: "Connecticut", 10: "Delaware",
        11: "District of Columbia", 12: "Florida", 13: "Georgia", 15: "Hawaii", 16: "Idaho", 17: "Illinois", 18: "Indiana", 19: "Iowa",
        20: "Kansas", 21: "Kentucky", 22: "Louisiana", 23: "Maine", 24: "Maryland", 25: "Massachusetts", 26: "Michigan", 27: "Minnesota",
        28: "Mississippi", 29: "Missouri", 30: "Montana", 31: "Nebraska", 32: "Nevada", 33: "New Hampshire", 34: "New Jersey",
        35: "New Mexico", 36: "New York", 37: "North Carolina", 38: "North Dakota", 39: "Ohio", 40: "Oklahoma", 41: "Oregon",
        42: "Pennsylvania", 44: "Rhode Island", 45: "South Carolina", 46: "South Dakota", 47: "Tennessee", 48: "Texas", 49: "Utah",
        50: "Vermont", 51: "Virginia", 53: "Washington", 54: "West Virginia", 55: "Wisconsin", 56: "Wyoming", 66: "Guam",
        72: "Puerto Rico", 78: "Virgin Islands"}
REGION = {}
for s in ["Connecticut", "Maine", "Massachusetts", "New Hampshire", "Rhode Island", "Vermont", "New Jersey", "New York", "Pennsylvania"]: REGION[s] = "Northeast"
for s in ["Illinois", "Indiana", "Michigan", "Ohio", "Wisconsin", "Iowa", "Kansas", "Minnesota", "Missouri", "Nebraska", "North Dakota", "South Dakota"]: REGION[s] = "Midwest"
for s in ["Delaware", "District of Columbia", "Florida", "Georgia", "Maryland", "North Carolina", "South Carolina", "Virginia", "West Virginia",
          "Alabama", "Kentucky", "Mississippi", "Tennessee", "Arkansas", "Louisiana", "Oklahoma", "Texas"]: REGION[s] = "South"
for s in ["Arizona", "Colorado", "Idaho", "Montana", "Nevada", "New Mexico", "Utah", "Wyoming", "Alaska", "California", "Hawaii", "Oregon", "Washington"]: REGION[s] = "West"
for s in ["Guam", "Puerto Rico", "Virgin Islands"]: REGION[s] = "Territories"
REGIONS_ALL = ["Northeast", "Midwest", "South", "West", "Territories"]

def age3(lower):
    return np.where(lower < 45, "18-44", np.where(lower < 65, "45-64", "65+"))

def _find(name):
    hits = []
    for p in [f"/kaggle/input/**/{name}", f"./{name}", f"./data/{name}"]:
        hits += [h for h in sorted(glob.glob(p, recursive=True)) if h not in hits]
    return hits

def load_kaggle():
    """heart_2022_no_nans.csv exactly as used by v2.1 (complete cases, 246,022 rows)."""
    hits = _find("heart_2022_no_nans.csv")
    if not hits: raise FileNotFoundError("Add the Kaggle dataset 'kamilpytlak/personal-key-indicators-of-heart-disease' as notebook input.")
    d = pd.read_csv(hits[0]); check("Kaggle file has 246,022 rows x 40 columns", d.shape == (246022, 40) or CFG3["QUICK"], d.shape)
    F = pd.DataFrame({"state": d["State"].astype(str), "Sex": d["Sex"].astype(str)})
    F["Age"] = age3(d["AgeCategory"].astype(str).str.extract(r"(\d+)")[0].astype(float).to_numpy())
    F["Race"] = d["RaceEthnicityCategory"].astype(str)
    for o in CFG3["OUTCOMES"]:
        v = d[o].astype(str).str.strip().str.lower()
        F[o] = np.where(v.str.startswith("yes"), 1.0, np.where(v.str.startswith("no"), 0.0, np.nan))
        if o == "HadDiabetes":   # yes_only coding (v2.1 primary): 'Yes' = 1; 'No', borderline and pregnancy-only = 0
            F[o] = np.where(v == "yes", 1.0, np.where(v.isin(["no", "no, pre-diabetes or borderline diabetes",
                                                               "yes, but only during pregnancy (female)"]), 0.0, np.nan))
    F["wdesign"] = 1.0
    other = [c for c in d.columns if c.startswith("Had")]
    feats = [c for c in d.columns if c not in other + ["State"]]
    num = [c for c in feats if pd.api.types.is_numeric_dtype(d[c])]; cat = [c for c in feats if c not in num]
    X = d[feats].copy()
    for c in cat: X[c] = X[c].astype(str)
    return F.reset_index(drop=True), X.reset_index(drop=True), num, cat, dict(source="Kaggle heart_2022_no_nans.csv", file=hits[0], n=len(d))

# ---- raw BRFSS: Kaggle-equivalent features from the CDC codebook variables ----
# Each feature lists candidate variable names (first present is used, so 2022 and 2023 names can differ); categorical
# features use the raw response code, with the listed codes treated as missing. Features that are missing in a year
# are dropped and logged. ChestScan is not used (not in the core questionnaire; a post-diagnostic marker per [28]).
RAW_FEATURES = {
    # name: (candidates, kind, missing codes or mapper)
    "Sex": (["SEXVAR", "_SEX"], "cat", []),
    "AgeCategory": (["_AGEG5YR"], "cat", [14]),
    "RaceEthnicityCategory": (["_IMPRACE", "_RACEPR1"], "cat", []),
    "GeneralHealth": (["GENHLTH"], "cat", [7, 9]),
    "PhysicalHealthDays": (["PHYSHLTH"], "days", None),
    "MentalHealthDays": (["MENTHLTH"], "days", None),
    "LastCheckupTime": (["CHECKUP1"], "cat", [7, 9]),
    "PhysicalActivities": (["EXERANY2"], "cat", [7, 9]),
    "SleepHours": (["SLEPTIM1"], "sleep", None),
    "RemovedTeeth": (["RMVTETH4"], "cat", [7, 9]),
    "DeafOrHardOfHearing": (["DEAF"], "cat", [7, 9]),
    "BlindOrVisionDifficulty": (["BLIND"], "cat", [7, 9]),
    "DifficultyConcentrating": (["DECIDE"], "cat", [7, 9]),
    "DifficultyWalking": (["DIFFWALK"], "cat", [7, 9]),
    "DifficultyDressingBathing": (["DIFFDRES"], "cat", [7, 9]),
    "DifficultyErrands": (["DIFFALON"], "cat", [7, 9]),
    "SmokerStatus": (["_SMOKER3"], "cat", [9]),
    "ECigaretteUsage": (["ECIGNOW2", "ECIGNOW3", "ECIGNOW1"], "cat", [7, 9]),
    "HeightInMeters": (["HTM4"], "scale100", None),
    "WeightInKilograms": (["WTKG3"], "scale100", None),
    "BMI": (["_BMI5"], "scale100", None),
    "AlcoholDrinkers": (["DRNKANY6", "DRNKANY5"], "cat", [7, 9]),
    "HIVTesting": (["HIVTST7"], "cat", [7, 9]),
    "FluVaxLast12": (["FLUSHOT7"], "cat", [7, 9]),
    "PneumoVaxEver": (["PNEUVAC4"], "cat", [7, 9]),
    "TetanusLast10Tdap": (["TETANUS1"], "cat", [7, 9]),
    "HighRiskLastYear": (["HIVRISK5"], "cat", [7, 9]),
    "CovidPos": (["COVIDPOS", "COVIDPO1"], "cat", [7, 9]),
}
RAW_OUTCOMES = {"HadHeartAttack": ["CVDINFR4"], "HadStroke": ["CVDSTRK3"], "HadDiabetes": ["DIABETE4"]}
RAW_DESIGN = {"wdesign": ["_LLCPWT"], "strata": ["_STSTR"], "psu": ["_PSU"]}

def _raw_file(year):
    """Local copy if attached as a Kaggle input (LLCP{year}.XPT or the zip); otherwise download from CDC (Internet ON)."""
    for nm in [f"LLCP_{year}.XPT", f"LLCP{year}.XPT", f"LLCP{year}.XPT ", f"LLCP{year}XPT.zip", f"llcp{year}.xpt", f"LLCP_{year}.xpt"]:
        h = _find(nm)
        if h: return h[0]
    dest = f"/kaggle/working/LLCP{year}XPT.zip" if os.path.isdir("/kaggle/working") else f"./LLCP{year}XPT.zip"
    if not os.path.exists(dest):
        print(f"downloading BRFSS {year} from CDC ...", flush=True)
        try:
            urllib.request.urlretrieve(CFG3["RAW_URLS"][year], dest)
        except Exception as e:
            raise RuntimeError(f"Could not download {CFG3['RAW_URLS'][year]} ({e}). In Kaggle: Settings -> Internet ON, or attach "
                               f"the file LLCP{year}XPT.zip as an input dataset.")
    return dest

def _read_xpt(path, keep):
    if path.lower().endswith(".zip"):
        z = zipfile.ZipFile(path); inner = [n for n in z.namelist() if n.strip().lower().endswith(".xpt")][0]
        path = z.extract(inner, os.path.dirname(path) or "."); path_clean = path.strip()
        if path_clean != path: os.replace(path, path_clean); path = path_clean
    parts = []
    for ch in pd.read_sas(path, format="xport", chunksize=50000):
        ch.columns = [c.strip().upper() for c in ch.columns]
        parts.append(ch[[c for c in keep if c in ch.columns]])
    return pd.concat(parts, ignore_index=True)

RAW_MAP_LOG = []
def load_raw(year, frame=None):
    """frame: optional pre-read DataFrame with raw CDC variable names (used by the offline test)."""
    keep = sorted({c for v in RAW_FEATURES.values() for c in v[0]} | {c for v in RAW_OUTCOMES.values() for c in v} |
                  {c for v in RAW_DESIGN.values() for c in v} | {"_STATE"})
    d = frame if frame is not None else _read_xpt(_raw_file(year), keep)
    n0 = len(d)
    def pick(cands):
        for c in cands:
            if c in d.columns: return c
        return None
    F = pd.DataFrame({"state": d["_STATE"].round().astype(int).map(FIPS)})
    agec = d["_AGEG5YR"].to_numpy(float); lower = np.where((agec >= 1) & (agec <= 13), 18 + np.maximum(0, agec - 1) * 5 + (agec > 1) * 2, np.nan)
    lower = np.where(agec == 1, 18, lower)                     # 1: 18-24, 2: 25-29, ..., 13: 80+
    F["Sex"] = np.where(d[pick(["SEXVAR", "_SEX"])] == 1, "Male", np.where(d[pick(["SEXVAR", "_SEX"])] == 2, "Female", "__MISSING__"))
    F["Age"] = np.where(np.isnan(lower), "__MISSING__", age3(np.nan_to_num(lower, nan=0)))
    rc = d[pick(["_IMPRACE", "_RACEPR1"])].to_numpy(float)
    F["Race"] = np.select([rc == 1, rc == 2, rc == 5, np.isin(rc, [3, 4, 6, 7])],
                          ["White, Non-Hispanic", "Black, Non-Hispanic", "Hispanic", "Other or multiracial, Non-Hispanic"], "__MISSING__")
    for o, cands in RAW_OUTCOMES.items():
        v = d[pick(cands)].to_numpy(float)
        F[o] = np.where(v == 1, 1.0, np.where(v == 2, 0.0, np.nan)) if o != "HadDiabetes" else \
               np.where(v == 1, 1.0, np.where(np.isin(v, [2, 3, 4]), 0.0, np.nan))   # yes_only, as v2.1
        RAW_MAP_LOG.append(dict(year=year, item=o, variable=pick(cands), kind="outcome"))
    for k, cands in RAW_DESIGN.items():
        c = pick(cands); F[k] = d[c].to_numpy(float) if c else np.nan
        RAW_MAP_LOG.append(dict(year=year, item=k, variable=c, kind="design"))
    X = pd.DataFrame(index=d.index); num, cat = [], []
    for name, (cands, kind, miss) in RAW_FEATURES.items():
        c = pick(cands)
        RAW_MAP_LOG.append(dict(year=year, item=name, variable=c, kind=kind))
        if c is None: continue
        v = d[c].to_numpy(float)
        if kind == "cat":
            X[name] = np.where(np.isnan(v) | np.isin(v, miss), "__MISSING__", pd.Series(v).round().astype("Int64").astype(str).to_numpy()); cat.append(name)
        elif kind == "days":
            X[name] = np.where(v == 88, 0.0, np.where((v >= 1) & (v <= 30), v, np.nan)); num.append(name)
        elif kind == "sleep":
            X[name] = np.where((v >= 1) & (v <= 24), v, np.nan); num.append(name)
        elif kind == "scale100":
            X[name] = np.where(v > 0, v / 100.0, np.nan); num.append(name)
    ok = F["state"].notna() & (F["Sex"] != "__MISSING__") & (F["Age"] != "__MISSING__") & (F["Race"] != "__MISSING__") & \
         F[CFG3["OUTCOMES"]].notna().any(axis=1) & (F["wdesign"] > 0)
    info = dict(source=f"CDC BRFSS LLCP{year}", n_raw=n0, n_kept=int(ok.sum()),
                n_dropped_missing_group_or_state=int((~ok).sum()),
                features_used=num + cat, features_missing=[n for n, (cands, _, _) in RAW_FEATURES.items() if pick(cands) is None])
    print(f"BRFSS {year}: {n0} raw rows, {int(ok.sum())} kept; features missing this year: {info['features_missing']}")
    return F[ok].reset_index(drop=True), X[ok].reset_index(drop=True), num, cat, info

# ---- NHANES August 2021-August 2023 (external validation with measured diabetes) ----
# Harmonised feature set available in BOTH surveys and free of post-diagnosis proxies: sex, 5-year age band (BRFSS
# _AGEG5YR codes 1-13), race/ethnicity (4 levels), measured (NHANES) or self-reported (BRFSS) height, weight and BMI, and
# smoking status (_SMOKER3 codes: 1 every day, 2 some days, 3 former, 4 never). Outcomes in NHANES (adults >= 18 with a
# valid HbA1c, phlebotomy weight WTPH2YR > 0): diagnosed = DIQ010 == 1 (told by a doctor; borderline = 0, as the BRFSS
# yes-only coding); measured = HbA1c >= 6.5%; total = diagnosed or measured; undiagnosed = measured and not diagnosed.
# Sensitivity: total_fpg also counts fasting plasma glucose >= 126 mg/dL in the fasting subsample (weight WTSAF2YR).
RACE4 = {"White": "White, Non-Hispanic", "Black": "Black, Non-Hispanic", "Hispanic": "Hispanic", "Other": "Other or multiracial, Non-Hispanic"}
HARM = ["Sex", "AgeCategory", "Race4", "SmokerStatus", "HeightInMeters", "WeightInKilograms", "BMI"]
HARM_NUM, HARM_CAT = ["HeightInMeters", "WeightInKilograms", "BMI"], ["Sex", "AgeCategory", "Race4", "SmokerStatus"]

def _nh_file(name):
    for nm in [f"{name}.xpt", f"{name}.XPT"]:
        h = _find(nm)
        if h: return h[0]
    dest = f"/kaggle/working/{name}.xpt" if os.path.isdir("/kaggle/working") else f"./{name}.xpt"
    if not os.path.exists(dest):
        try: urllib.request.urlretrieve(CFG3["NHANES_URL"].format(name), dest)
        except Exception as e:
            raise RuntimeError(f"NHANES file {name}.xpt not found. Attach the six NHANES 2021-2023 files ({', '.join(CFG3['NHANES_FILES'])}) "
                               f"as a Kaggle input dataset, or switch Internet ON ({e}).")
    return dest

def load_nhanes(frames=None):
    """frames: optional dict name -> DataFrame (offline test)."""
    T = {}
    for nm in CFG3["NHANES_FILES"]:
        d = frames[nm] if frames is not None else pd.read_sas(_nh_file(nm), format="xport")
        d.columns = [c.strip().upper() for c in d.columns]; T[nm] = d.set_index("SEQN")
    D = T["DEMO_L"].join([T[k] for k in CFG3["NHANES_FILES"] if k != "DEMO_L"], how="left")
    D = D[(D["RIDAGEYR"] >= 18) & (D.get("WTPH2YR", pd.Series(0, index=D.index)).fillna(0) > 0) & D["LBXGH"].notna()]
    age = D["RIDAGEYR"].to_numpy(float)
    code = np.where(age >= 80, 13, np.where(age < 25, 1, np.floor((age - 25) / 5) + 2)).astype(int)
    X = pd.DataFrame(index=D.index)
    X["Sex"] = D["RIAGENDR"].round().astype("Int64").astype(str).to_numpy()
    X["AgeCategory"] = code.astype(str)
    r = D["RIDRETH3"].to_numpy(float)
    X["Race4"] = np.select([r == 3, r == 4, np.isin(r, [1, 2]), np.isin(r, [6, 7])], ["White", "Black", "Hispanic", "Other"], "__MISSING__")
    s20, s40 = D["SMQ020"].to_numpy(float), D["SMQ040"].to_numpy(float)
    X["SmokerStatus"] = np.select([s20 == 2, (s20 == 1) & (s40 == 1), (s20 == 1) & (s40 == 2), (s20 == 1) & (s40 == 3)], ["4", "1", "2", "3"], "__MISSING__")
    X["HeightInMeters"] = D["BMXHT"].to_numpy(float) / 100; X["WeightInKilograms"] = D["BMXWT"].to_numpy(float); X["BMI"] = D["BMXBMI"].to_numpy(float)
    F = pd.DataFrame(index=D.index)
    F["Sex"] = np.where(D["RIAGENDR"] == 1, "Male", "Female"); F["Age"] = age3(age); F["Race"] = X["Race4"].map(RACE4).fillna("__MISSING__").to_numpy()
    dq = D["DIQ010"].to_numpy(float)
    F["diagnosed"] = np.where(dq == 1, 1.0, np.where(np.isin(dq, [2, 3]), 0.0, np.nan))
    F["measured"] = (D["LBXGH"].to_numpy(float) >= CFG3["HBA1C_CUT"]).astype(float)
    F["total"] = np.where(np.isnan(F["diagnosed"]), np.nan, np.maximum(F["diagnosed"], F["measured"]))
    F["undiagnosed"] = np.where(np.isnan(F["diagnosed"]), np.nan, ((F["measured"] == 1) & (F["diagnosed"] == 0)).astype(float))
    glu = D["LBXGLU"].to_numpy(float) if "LBXGLU" in D else np.full(len(D), np.nan)
    F["fasting"] = (D.get("WTSAF2YR", pd.Series(0, index=D.index)).fillna(0).to_numpy() > 0) & ~np.isnan(glu)
    F["total_fpg"] = np.where(F["fasting"], np.maximum(F["total"], (glu >= CFG3["FPG_CUT"]).astype(float)), np.nan)
    F["undiag_fpg"] = np.where(F["fasting"], ((F["total_fpg"] == 1) & (F["diagnosed"] == 0)).astype(float), np.nan)
    F["wdesign"] = D["WTPH2YR"].to_numpy(float); F["wfast"] = D.get("WTSAF2YR", pd.Series(np.nan, index=D.index)).to_numpy(float)
    F["strata"] = D["SDMVSTRA"].to_numpy(float); F["psu"] = D["SDMVSTRA"].to_numpy(float) * 10 + D["SDMVPSU"].to_numpy(float)
    ok = (F["Race"] != "__MISSING__") & F["diagnosed"].notna()
    info = dict(source="NHANES August 2021-August 2023", n_adults_with_hba1c=int(len(D)), n_kept=int(ok.sum()),
                prevalence_diagnosed=float(np.average(F.loc[ok, "diagnosed"], weights=F.loc[ok, "wdesign"])),
                prevalence_total=float(np.average(F.loc[ok, "total"], weights=F.loc[ok, "wdesign"])),
                prevalence_undiagnosed=float(np.average(F.loc[ok, "undiagnosed"], weights=F.loc[ok, "wdesign"])))
    print("NHANES:", info)
    return F[ok].reset_index(drop=True), X[ok].reset_index(drop=True), info

def brfss_harmonised(Fr, Xr):
    """BRFSS raw frame -> the harmonised NHANES feature set (Race4 from the standard Race column)."""
    X = pd.DataFrame({"Sex": Xr["Sex"].to_numpy(), "AgeCategory": Xr["AgeCategory"].to_numpy(),
                      "Race4": Fr["Race"].map({v: k for k, v in RACE4.items()}).fillna("__MISSING__").to_numpy(),
                      "SmokerStatus": Xr["SmokerStatus"].to_numpy(), "HeightInMeters": Xr["HeightInMeters"].to_numpy(),
                      "WeightInKilograms": Xr["WeightInKilograms"].to_numpy(), "BMI": Xr["BMI"].to_numpy()})
    return X


## 3. Methods

In [ ]:
# ======================= 3. Models, conformal methods, metrics and the exact worst-group reference =======================
# weighted_q, conformal_sets, sage_sets, kish, _qr_lp, fit_domain, weights_from and the reference are VERBATIM from v2.1.
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from scipy.optimize import linprog
from scipy.sparse import csr_matrix, hstack as sp_hstack, identity as sp_identity
from scipy.stats import wilcoxon, binomtest

MODEL_BACKEND = {}
def make_model(name, seed):
    """Same settings as v2.1 (Table S4)."""
    if name == "LR":
        MODEL_BACKEND[name] = "sklearn LogisticRegression"
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, C=1.0))
    try:
        import lightgbm as lgb; MODEL_BACKEND[name] = f"lightgbm {lgb.__version__}"
        return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.8, random_state=seed, verbose=-1)
    except ImportError:
        MODEL_BACKEND[name] = "FALLBACK sklearn HistGradientBoosting"
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=seed)

def make_encoder(num, cat):
    return ColumnTransformer([("num", SimpleImputer(strategy="median"), num),
                              ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), cat)],
                             remainder="drop", verbose_feature_names_out=False)

def weighted_q(cal_scores, cal_w, test_w, alpha):
    o = np.argsort(cal_scores); s = cal_scores[o]; cw = np.cumsum(cal_w[o]); W = cw[-1]
    k = np.searchsorted(cw, (1 - alpha) * (W + test_w), side="left")
    return np.where(k < len(s), s[np.minimum(k, len(s) - 1)], np.inf)

def conformal_sets(p_cal, y_cal, p_te, alpha, w_cal=None, w_te=None, class_cond=False):
    n_te = len(p_te)
    w_cal = np.ones(len(p_cal)) if w_cal is None else w_cal
    w_te = np.ones(n_te) if w_te is None else w_te
    P_te, P_cal = np.c_[1 - p_te, p_te], np.c_[1 - p_cal, p_cal]
    S = np.zeros((n_te, 2), bool)
    if class_cond:
        for c in (0, 1):
            m = y_cal == c
            S[:, c] = (1 - P_te[:, c]) <= (weighted_q(1 - P_cal[m, c], w_cal[m], w_te, alpha) if m.any() else np.inf)
    else:
        q = weighted_q(1 - P_cal[np.arange(len(y_cal)), y_cal], w_cal, w_te, alpha)
        S = (1 - P_te) <= q[:, None]
    return S

def sage_sets(p_cal, y_cal, keys_cal, p_te, keys_te, alpha, w_cal=None, w_te=None, min_ess=100):
    n_te = len(p_te)
    w_cal = np.ones(len(p_cal)) if w_cal is None else w_cal
    w_te = np.ones(n_te) if w_te is None else w_te
    P_te, P_cal = np.c_[1 - p_te, p_te], np.c_[1 - p_cal, p_cal]
    S = np.zeros((n_te, 2), bool)
    for c in (0, 1):
        mc = y_cal == c; done = np.zeros(n_te, bool)
        for L, (kc, kt) in enumerate(zip(keys_cal, keys_te)):
            last = L == len(keys_cal) - 1
            for k in np.unique(kt[~done]):
                ti = np.where(~done & (kt == k))[0]
                cm = mc & (kc == k); wc = w_cal[cm]
                ess = wc.sum() ** 2 / (wc ** 2).sum() if cm.any() else 0.0
                if ess < min_ess and not last: continue
                q = weighted_q(1 - P_cal[cm, c], wc, w_te[ti], alpha) if cm.any() else np.full(len(ti), np.inf)
                S[ti, c] = (1 - P_te[ti, c]) <= q
                done[ti] = True
    return S

def kish(w):
    w = np.asarray(w, float)
    return float(w.sum() ** 2 / (w ** 2).sum()) if len(w) and (w ** 2).sum() > 0 else 0.0

def _qr_lp(Xd, s, w, tau):
    n, d = Xd.shape
    c = np.r_[np.zeros(d), tau * w, (1 - tau) * w]
    A = sp_hstack([csr_matrix(Xd), sp_identity(n), -sp_identity(n)]).tocsr()
    res = linprog(c, A_eq=A, b_eq=s, bounds=[(None, None)] * d + [(0, None)] * (2 * n), method="highs")
    if not res.success: raise RuntimeError("quantile-regression LP failed: " + res.message)
    return res.x[:d]

def fit_domain(Xs, Xt):
    Xd = np.r_[Xs, Xt]; yd = np.r_[np.zeros(len(Xs)), np.ones(len(Xt))]
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, C=CFG3["DOMAIN_C"])).fit(Xd, yd)
    m.log_prior_ = np.log(len(Xs) / len(Xt))
    return m

def weights_from(m, Xq):
    return np.exp(np.clip(m.decision_function(Xq) + m.log_prior_, -30, 30))

# ---------------- tuned conditional calibration (Gibbs, Cherian & Candes, JRSSB 2025) ----------------
# Function class F = span of indicator features: intercept, Sex x Age cells, optionally race, optionally B bins of the
# estimated density ratio (bins = quantiles on s_dom, label-free). For a finite indicator class the augmented
# quantile-regression threshold g_S(x) depends on x only through its pattern (cell, race, bin). We use the
# deterministic conservative form: the test score is imputed at its maximum (S = 1 for the LAC score), so the set
# {y : S(x, y) <= g_1(x)} contains the exact non-randomised Gibbs set and inherits its coverage guarantee
# (P(Y in C(X) | X in group) >= 1 - alpha for every group in F; weighted versions for shifts in the span of F).
# Fitted per label (label-conditional), one LP per (label, pattern present in the evaluation data).
def _patterns(G, cols):
    return pd.Series(list(map(tuple, G[cols].to_numpy())))

def gibbs_thresholds(s_cal, D_cal, D_pat, tau, grid):
    """s_cal: calibration scores of one label; D_cal: their indicator rows; D_pat: one indicator row per pattern.
    Returns threshold per pattern (augmented fit with the pattern's test point at score 1)."""
    s = np.ceil(s_cal / grid) * grid
    key = pd.DataFrame(D_cal).astype(np.int8).astype(str).agg("".join, axis=1) + "|" + pd.Series(s).round(6).astype(str)
    comp = pd.DataFrame({"k": key, "i": np.arange(len(s))}).groupby("k")["i"].agg(["first", "size"])
    Dc, sc, wc = D_cal[comp["first"].to_numpy()], s[comp["first"].to_numpy()], comp["size"].to_numpy(float)
    act = D_cal.sum(0) > 0
    th = np.empty(len(D_pat))
    for j, dp in enumerate(D_pat):
        a = act | (dp > 0)
        beta = _qr_lp(np.r_[Dc, dp[None, :]][:, a], np.r_[sc, 1.0], np.r_[wc, 1.0], tau)
        th[j] = float(dp[a] @ beta)
    return th

def gibbs_sets(p_cal, y_cal, Gc, p_ev_list, G_ev_list, cols, alpha, grid):
    """cols: indicator columns of the function class (Gc / G_ev: DataFrames of 0/1 indicators incl. 'one')."""
    outs = [np.zeros((len(p), 2), bool) for p in p_ev_list]
    pats = pd.concat([G[cols] for G in G_ev_list]).drop_duplicates().to_numpy(float)
    pidx = {tuple(r): j for j, r in enumerate(pats)}
    for c in (0, 1):
        m = y_cal == c; n = int(m.sum())
        if n == 0:
            for S in outs: S[:, c] = True
            continue
        tau = 1 - alpha                                    # augmented fit: no finite-sample correction needed
        sc = 1 - (p_cal[m] if c == 1 else 1 - p_cal[m])
        th = gibbs_thresholds(sc, Gc[cols].to_numpy(float)[m], pats, tau, grid)
        for S, p, G in zip(outs, p_ev_list, G_ev_list):
            j = np.array([pidx[tuple(r)] for r in G[cols].to_numpy(float)])
            sc_ev = 1 - (p if c == 1 else 1 - p)
            S[:, c] = sc_ev <= th[j] + 1e-9
    return outs

# ---------------- metrics (optionally weighted by the BRFSS design weight) ----------------
def _wmean(x, w):
    return float(np.sum(w * x) / np.sum(w)) if len(x) and np.sum(w) > 0 else np.nan

def metrics(S, y, G, w=None, eligible=None):
    """Coverage, positive-class coverage, set size, share of {0,1}, empty and singleton-{1} rates, and the minimum
    positive-class coverage over ELIGIBLE groups (Sex, Age, SexxAge: 'cal'; Race: 'race'). Eligibility (MIN_GROUP rows and
    MIN_POS positives in the evaluation set) is decided on counts, so weighted and unweighted metrics use the same groups."""
    w = np.ones(len(y)) if w is None else w
    cov = S[np.arange(len(y)), y].astype(float); size = S.sum(1); pos = y == 1
    out = dict(n=len(y), n_pos=int(pos.sum()), coverage=_wmean(cov, w), cov_pos=_wmean(cov[pos], w[pos]),
               cov_neg=_wmean(cov[~pos], w[~pos]), set_size=_wmean(size, w), both=_wmean((size == 2).astype(float), w),
               empty=_wmean((size == 0).astype(float), w), single1=_wmean((S[:, 1] & ~S[:, 0]).astype(float), w),
               ppv_single1=_wmean(y[S[:, 1] & ~S[:, 0]].astype(float), w[S[:, 1] & ~S[:, 0]]))
    out["absdev_cov_pos"] = abs(out["cov_pos"] - (1 - ALPHA)) if not np.isnan(out["cov_pos"]) else np.nan
    for name, axes in [("cal", ["Sex", "Age", "SexxAge"]), ("race", ["Race"])]:
        vals = []
        for ax in axes:
            for g, ix in G.groupby(ax).indices.items():
                if len(ix) >= CFG3["MIN_GROUP"] and int(y[ix].sum()) >= CFG3["MIN_POS"]:
                    pi = ix[y[ix] == 1]; vals.append(_wmean(cov[pi], w[pi]))
        out[f"wg_{name}_pos"] = float(np.min(vals)) if vals else np.nan
        out[f"n_elig_{name}"] = len(vals)
    return out

def _beta_pi(ncell, alpha, sims, rng):
    n = int(ncell)
    if n <= 0: return np.ones(sims)
    k = math.ceil((1 - alpha) * (n + 1))
    return np.ones(sims) if k > n else rng.beta(k, n + 1 - k, size=sims)

def exact_reference(y_ev, G_ev, cal_n, alpha, sims, rng):
    """Proposition 3: law of the minimum positive-class coverage over eligible calibrated groups under label x cell split
    CP with cal_n[cell] calibration positives (exact for continuous scores; cells without positives use the Age parent).
    Returns mean and 5th percentile."""
    pos = np.where(y_ev == 1)[0]
    if not len(pos): return np.nan, np.nan
    cell = G_ev["SexxAge"].to_numpy(str)[pos]
    def n_eff(c):
        v = cal_n.get(c, 0)
        return v if v > 0 else sum(x for k, x in cal_n.items() if k.split("|")[-1] == c.split("|")[-1])
    cells = sorted(set(cell)); ci = np.searchsorted(np.array(cells), cell)
    pi = np.column_stack([_beta_pi(n_eff(c), alpha, sims, rng) for c in cells])[:, ci]
    M = []
    for ax in ["Sex", "Age", "SexxAge"]:
        for g, ix in G_ev.groupby(ax).indices.items():
            if len(ix) >= CFG3["MIN_GROUP"] and int(y_ev[ix].sum()) >= CFG3["MIN_POS"]:
                v = np.zeros(len(y_ev), bool); v[ix] = True; M.append(v[pos])
    if not M: return np.nan, np.nan
    M = np.array(M, np.float32).T
    Z = (rng.random((sims, len(pos))) < pi).astype(np.float32)
    mn = ((Z @ M) / M.sum(0)).min(1)
    return float(mn.mean()), float(np.quantile(mn, 0.05))

def gibbs_unaug_sets(p_cal, y_cal, Gc, p_ev, G_ev, cols, alpha, grid):
    """Unaugmented (split) fit used ONLY to select the function class on source data: one LP per label at level
    (1-alpha)(n+1)/n; threshold = D(x) beta."""
    S = np.zeros((len(p_ev), 2), bool)
    for c in (0, 1):
        m = y_cal == c; n = int(m.sum())
        tau = 1.0 if n == 0 else min(1.0, (1 - alpha) * (n + 1) / n)
        if tau >= 1.0: S[:, c] = True; continue
        sc = np.ceil((1 - (p_cal[m] if c == 1 else 1 - p_cal[m])) / grid) * grid
        D = Gc[cols].to_numpy(float)[m]
        key = pd.DataFrame(D).astype(np.int8).astype(str).agg("".join, axis=1) + "|" + pd.Series(sc).round(6).astype(str)
        comp = pd.DataFrame({"k": key, "i": np.arange(len(sc))}).groupby("k")["i"].agg(["first", "size"])
        act = D.sum(0) > 0; beta = np.zeros(D.shape[1])
        beta[act] = _qr_lp(D[comp["first"].to_numpy()][:, act], sc[comp["first"].to_numpy()], comp["size"].to_numpy(float), tau)
        S[:, c] = (1 - (p_ev if c == 1 else 1 - p_ev)) <= G_ev[cols].to_numpy(float) @ beta + 1e-9
    return S

# ======================= v4 additions =======================
from sklearn.calibration import CalibratedClassifierCV
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
_make_model_v3 = make_model
def make_model(name, seed):
    """v3 models plus CBCAL: CatBoost (500 trees, depth 6) with isotonic calibration (3-fold, on s_train only)."""
    if name != "CBCAL": return _make_model_v3(name, seed)
    try:
        from catboost import CatBoostClassifier; import catboost
        base = CatBoostClassifier(iterations=500, depth=6, learning_rate=0.05, random_seed=seed, verbose=0, thread_count=-1)
        MODEL_BACKEND[name] = f"catboost {catboost.__version__} + isotonic"
    except ImportError:
        base = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=seed)
        MODEL_BACKEND[name] = "FALLBACK sklearn HistGradientBoosting + isotonic"
    return CalibratedClassifierCV(base, method="isotonic", cv=3)

def make_encoder_v(num, cat, variant="median"):
    """median: v3 encoder. misind: median imputation plus a missing indicator for each numeric feature with NaNs."""
    if variant == "misind":
        return ColumnTransformer([("num", SimpleImputer(strategy="median", add_indicator=True), num),
                                  ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), cat)],
                                 remainder="drop", verbose_feature_names_out=False)
    return make_encoder(num, cat)

class MIEncoder:
    """Multiple imputation by chained equations for the numeric features (Bayesian ridge, posterior draws), with the
    one-hot categorical features (missing = its own level) as predictors. The outcome is NOT used, because it is unknown
    when a prediction set is formed. Fitted on (a random subset of) s_train only; draw j uses random_state = j."""
    def __init__(self, num, cat, j, fit_rows, max_iter):
        self.num, self.cat, self.j, self.fit_rows, self.max_iter = num, cat, j, fit_rows, max_iter
    def fit(self, X):
        self.oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32).fit(X[self.cat])
        Z = self._z(X); rs = np.random.default_rng(self.j)
        sub = Z if len(Z) <= self.fit_rows else Z[rs.choice(len(Z), self.fit_rows, replace=False)]
        self.imp = IterativeImputer(sample_posterior=True, max_iter=self.max_iter, random_state=self.j, n_nearest_features=15).fit(sub)
        return self
    def _z(self, X): return np.c_[X[self.num].to_numpy(float), self.oh.transform(X[self.cat])]
    def transform(self, X): return self.imp.transform(self._z(X)).astype(np.float32)

def metrics4(S, y, G, w=None):
    """v3 metrics plus the triage quantities: share ruled out ({0}), missed positives inside {0} and NPV of {0}."""
    out = metrics(S, y, G, w)
    w = np.ones(len(y)) if w is None else w
    zero = S[:, 0] & ~S[:, 1]
    out.update(zero=_wmean(zero.astype(float), w), miss0=_wmean((zero & (y == 1)).astype(float), w),
               npv0=_wmean((y[zero] == 0).astype(float), w[zero]))
    return out

def cutoff_rules(p_cal, y_cal, p_ev, y_ev, w):
    """Single risk cut-offs chosen on the calibration data: sensitivity 0.90 (10th percentile of the positives' risk),
    and Youden's J. Per 1,000 evaluated adults (design-weighted): flagged, ruled out, missed positives, PPV, NPV."""
    from sklearn.metrics import roc_curve
    t90 = float(np.quantile(p_cal[y_cal == 1], ALPHA)) if (y_cal == 1).any() else 0.5
    fpr, tpr, th = roc_curve(y_cal, p_cal); tY = float(th[np.argmax(tpr - fpr)])
    res = {}
    for nm, t in [("cut_sens90", t90), ("cut_youden", tY)]:
        f = p_ev >= t
        res[nm] = dict(threshold=t, flagged=_wmean(f.astype(float), w), ruled_out=_wmean((~f).astype(float), w),
                       missed=_wmean((~f & (y_ev == 1)).astype(float), w), ppv=_wmean(y_ev[f].astype(float), w[f]),
                       npv=_wmean((y_ev[~f] == 0).astype(float), w[~f]), sens=_wmean(f[y_ev == 1].astype(float), w[y_ev == 1]))
    return res

def rw_multipliers(strata, psu, psu_by_stratum, B, seed):
    """Rao-Wu rescaled bootstrap: in each stratum with n_h PSUs draw n_h - 1 PSUs with replacement; multiplier
    n_h/(n_h - 1) x count. Draws are seeded by stratum, so every analysis of the same respondents uses the same
    replicates (paired contrasts) and replicates are independent across strata (and so across states)."""
    M = np.ones((B, len(psu)), np.float32)
    for h in np.unique(strata):
        ix = np.where(strata == h)[0]; P = psu_by_stratum[h]; nh = len(P)
        if nh < 2: continue
        cnt = np.random.default_rng([seed, int(h)]).multinomial(nh - 1, np.full(nh, 1 / nh), size=B) * (nh / (nh - 1))
        col = np.searchsorted(P, psu[ix]); M[:, ix] = cnt[:, col]
    return M

def rep_metrics(S, y, G, W):
    """Design-weighted metrics under each replicate weight vector (rows of W); eligibility fixed by the full sample."""
    cov = S[np.arange(len(y)), y].astype(float); size = S.sum(1).astype(float); pos = y == 1
    r = dict(cov_pos=(W[:, pos] @ cov[pos]) / W[:, pos].sum(1), set_size=(W @ size) / W.sum(1),
             both=(W @ (size == 2).astype(float)) / W.sum(1))
    r["absdev_cov_pos"] = np.abs(r["cov_pos"] - (1 - ALPHA))
    vals = []
    for ax in ["Sex", "Age", "SexxAge"]:
        for g, ix in G.groupby(ax).indices.items():
            if len(ix) >= CFG3["MIN_GROUP"] and int(y[ix].sum()) >= CFG3["MIN_POS"]:
                pi = ix[y[ix] == 1]; vals.append((W[:, pi] @ cov[pi]) / W[:, pi].sum(1))
    r["wg_cal_pos"] = np.min(np.array(vals), 0) if vals else np.full(len(W), np.nan)
    return r


## 4. Engine

In [ ]:
# ======================= 4. Engine v4 (same disjoint partitions and train-only preprocessing as v3) =======================
UNIT_DIR, DONE_JSON = f"{OUT}/units", f"{OUT}/done_units.json"; os.makedirs(UNIT_DIR, exist_ok=True)
DONE = set(map(tuple, json.load(open(DONE_JSON)))) if os.path.exists(DONE_JSON) else set()
STUDY_IX = {s: i for i, s in enumerate(["RAWP", "RAWP_MISIND", "RAWP_MI", "RAWP_FULL", "RAWP_CB", "LOSO_G", "WEST", "NHANES1", "NHANES2"])}
PSU_BY_STRATUM = {}   # filled in Section 5 from the full file (all respondents of each stratum)

def _save(rows, path):
    if rows: pd.DataFrame(rows).to_csv(path, index=False)

def _cap(idx, n, rng): return idx if len(idx) <= n else rng.choice(idx, n, replace=False)

def gframe(F, rows):
    G = F.iloc[rows][["Sex", "Age", "Race"]].reset_index(drop=True).astype(str)
    G["SexxAge"] = G["Sex"] + "|" + G["Age"]; return G

SA_CATS = [f"{s}|{a}" for s in ["Female", "Male"] for a in ["18-44", "45-64", "65+"]]
def indicators(G, wraw, race_cats, edges):
    D = pd.DataFrame({f"SA={c}": (G["SexxAge"] == c).astype(float).to_numpy() for c in SA_CATS})
    for r in race_cats[1:]: D[f"R={r}"] = (G["Race"] == r).astype(float).to_numpy()
    if edges is not None:
        b = np.digitize(wraw, edges)
        for k in range(1, len(edges) + 1): D[f"W={k}"] = (b == k).astype(float)
    return D

def _mark_done(key, tag, rows, extra=None):
    _save(rows, f"{UNIT_DIR}/{tag}.csv")
    for suffix, r in (extra or {}).items(): _save(r, f"{UNIT_DIR}/{tag}.{suffix}.csv")
    DONE.add(key); json.dump(sorted(map(list, DONE)), open(DONE_JSON, "w"))

def build_sets(pc, yc, kc, pe, ke, wc, we, pu=None, yu=None, ku=None):
    SETS = {"ClassCond": conformal_sets(pc, yc, pe, ALPHA, class_cond=True),
            "GroupCond": sage_sets(pc, yc, kc, pe, ke, ALPHA, min_ess=CFG3["MIN_ESS"])}
    if wc is not None:
        SETS["Weighted+ClassCond"] = conformal_sets(pc, yc, pe, ALPHA, w_cal=wc, w_te=we, class_cond=True)
        SETS["SAGE-CP"] = sage_sets(pc, yc, kc, pe, ke, ALPHA, w_cal=wc, w_te=we, min_ess=CFG3["MIN_ESS"])
    if pu is not None:
        SETS["TargetOracle"] = sage_sets(pu, yu, ku, pe, ke, ALPHA, min_ess=CFG3["MIN_ESS"])
    for a2 in CFG3["EXTRA_ALPHAS"]:
        SETS[f"ClassCond@{a2:.2f}"] = conformal_sets(pc, yc, pe, a2, class_cond=True)
        SETS[f"GroupCond@{a2:.2f}"] = sage_sets(pc, yc, kc, pe, ke, a2, min_ess=CFG3["MIN_ESS"])
        if wc is not None:
            SETS[f"SAGE-CP@{a2:.2f}"] = sage_sets(pc, yc, kc, pe, ke, a2, w_cal=wc, w_te=we, min_ess=CFG3["MIN_ESS"])
    return SETS

def gibbs_tuned(pc, yc, g, w, pr, y, race_cats, WR_dom, rng, glog, tagd):
    GB = CFG3["GIBBS"]; tg = time.time()
    sub = np.r_[_cap(np.where(yc == 0)[0], GB["n_cal_max"], rng), _cap(np.where(yc == 1)[0], GB["n_cal_max"], rng)]
    cand = []
    for B in GB["wbins"]:
        edges = None if B == 0 else np.quantile(WR_dom, np.linspace(0, 1, B + 1)[1:-1])
        for use_race in GB["race"]:
            rc = race_cats if use_race else race_cats[:1]
            Dc = indicators(g["s_cal"].iloc[sub].reset_index(drop=True), w["s_cal"][1][sub], rc, edges)
            Ds = indicators(g["s_test"], w["s_test"][1], rc, edges); cols = list(Dc.columns)
            S_st = gibbs_unaug_sets(pc[sub], yc[sub], Dc, pr["s_test"], Ds, cols, ALPHA, GB["score_grid"])
            est = metrics(S_st, y["s_test"], g["s_test"], w=w["s_test"][1])
            short = max(0.0, (1 - ALPHA) - est["wg_cal_pos"]) if not np.isnan(est["wg_cal_pos"]) else np.inf
            cand.append(dict(B=B, race=use_race, edges=edges, cols=cols, short=short, size=est["set_size"]))
            glog.append(dict(**tagd, wbins=B, race=use_race, n_cols=len(cols), est_wg_cal_pos=est["wg_cal_pos"],
                             est_shortfall=short, est_set_size=est["set_size"]))
    best = min(c["short"] for c in cand)
    sel = min([c for c in cand if c["short"] <= best + GB["tie_tol"]], key=lambda c: c["size"])
    rc = race_cats if sel["race"] else race_cats[:1]
    Dc = indicators(g["s_cal"].iloc[sub].reset_index(drop=True), w["s_cal"][1][sub], rc, sel["edges"])
    De = indicators(g["t_eval"], w["t_eval"][1], rc, sel["edges"])
    S = gibbs_sets(pc[sub], yc[sub], Dc, [pr["t_eval"]], [De], sel["cols"], ALPHA, GB["score_grid"])[0]
    for r in glog[-len(cand):]: r["selected"] = (r["wbins"] == sel["B"]) and (r["race"] == sel["race"])
    return S, dict(gibbs_wbins=sel["B"], gibbs_race=sel["race"], gibbs_seconds=time.time() - tg)

def run_unit4(study, unit, seed, Fs, Xs, src, Ft, Xt, tgt, num, cat, unit_index=0, variant="median", gibbs=False, boot=False,
              src_outcomes=None, eval_outcomes=None, cutoffs=False, eval_by=None):
    """Leave-one-unit-out run (v3 partitions). variant: median | misind | mi (m = MI_M imputations, results per imputation).
    src_outcomes/eval_outcomes: outcome used to train and calibrate, and the outcome(s) it is evaluated against
    (NHANES1: train on BRFSS self-report, evaluate on NHANES diagnosed and total diabetes)."""
    key = (study, str(unit), int(seed))
    if key in DONE: return "done"
    t0 = time.time(); rng = np.random.default_rng(seed * 1_000_003 + unit_index * 7919 + 17 * STUDY_IX[study])
    src = rng.permutation(src); tgt = rng.permutation(tgt)
    n_cal, n_st, n_dom = (min(CFG3[k], len(src) // 5) for k in ["N_CAL", "N_SRC_TEST", "N_DOM"])
    a = 0; P = {}
    P["s_cal"] = src[a:a + n_cal]; a += n_cal; P["s_test"] = src[a:a + n_st]; a += n_st; P["s_dom"] = src[a:a + n_dom]; a += n_dom
    P["s_train"] = _cap(src[a:], CFG3["MAX_TRAIN"], rng)
    nu = min(len(tgt) // 2, 20000) if eval_by is not None else len(tgt) // 2
    P["t_unlab"], P["t_eval"] = tgt[:nu], tgt[nu:]
    if Ft is Fs: assert not np.isin(P["t_eval"], np.r_[P["s_cal"], P["s_test"], P["s_dom"], P["s_train"], P["t_unlab"]]).any()
    SRC_PARTS = ["s_cal", "s_test", "s_dom", "s_train"]
    Xof = lambda k: (Xs if k in SRC_PARTS else Xt).iloc[P[k]]
    Fof = lambda k: (Fs if k in SRC_PARTS else Ft)
    enc_d = make_encoder(num, cat).fit(pd.concat([Xof("s_dom"), Xof("t_unlab")]))
    XD = {k: enc_d.transform(Xof(k)).astype(np.float32) for k in ["s_cal", "s_test", "s_dom", "t_unlab", "t_eval"]}
    dm = fit_domain(XD["s_dom"], XD["t_unlab"]); WR = {k: weights_from(dm, XD[k]) for k in XD}
    clipv = float(np.quantile(WR["s_dom"], CFG3["CLIP_Q"])); WC = {k: np.minimum(v, clipv) for k, v in WR.items()}
    G = {k: gframe(Fof(k), P[k]) for k in ["s_cal", "s_test", "t_unlab", "t_eval"]}
    race_cats = sorted(pd.unique(pd.concat([G["s_cal"]["Race"], G["t_eval"]["Race"]])))
    imps = range(CFG3["MI_M"]) if variant == "mi" else [None]
    rows, glog, brows, crow = [], [], [], []
    src_outcomes = src_outcomes or CFG3["OUTCOMES"]
    for j in imps:
        if variant == "mi":
            enc_b = MIEncoder(num, cat, j, CFG3["MI_FIT_ROWS"], CFG3["MI_MAX_ITER"]).fit(Xof("s_train"))
        else:
            enc_b = make_encoder_v(num, cat, variant).fit(Xof("s_train"))
        XB = {k: enc_b.transform(Xof(k)).astype(np.float32) for k in ["s_train", "s_cal", "s_test", "t_unlab", "t_eval"]}
        for oi, o in enumerate(src_outcomes):
            ev_list = (eval_outcomes or {}).get(o, [o])
            yv = {k: Fof(k)[o if k in SRC_PARTS else ev_list[0]].to_numpy()[P[k]] for k in P}
            ok = {k: ~np.isnan(yv[k]) for k in P}
            y = {k: yv[k][ok[k]].astype(int) for k in P}
            if y["t_eval"].sum() < 1 or y["s_cal"].sum() < 1: continue
            for mi, mname in enumerate(CFG3["MODELS"][study.replace("NHANES1", "NHANES")]):
                model = make_model(mname, seed).fit(XB["s_train"][ok["s_train"]], y["s_train"])
                pr = {k: model.predict_proba(XB[k][ok[k]])[:, 1] for k in ["s_cal", "s_test", "t_unlab", "t_eval"]}
                g = {k: G[k][ok[k]].reset_index(drop=True) for k in G}
                w = {k: (WC[k][ok[k]], WR[k][ok[k]]) for k in ["s_cal", "s_test", "t_unlab", "t_eval"]}
                keys = lambda k: [g[k]["SexxAge"].to_numpy(str), g[k]["Age"].to_numpy(str), np.array(["all"] * len(g[k]))]
                pc, yc = pr["s_cal"], y["s_cal"]
                same_label = ev_list[0] == o
                SETS = build_sets(pc, yc, keys("s_cal"), pr["t_eval"], keys("t_eval"), w["s_cal"][0], w["t_eval"][0],
                                  *( (pr["t_unlab"], y["t_unlab"], keys("t_unlab")) if same_label else (None, None, None)))
                gib_info = {}
                if gibbs and mname == "LR":
                    SETS["Gibbs-tuned"], gib_info = gibbs_tuned(pc, yc, g, w, pr, y, race_cats, WR["s_dom"], rng, glog,
                                                                dict(study=study, unit=unit, seed=seed, outcome=o, model=mname))
                cal_n = pd.Series(g["s_cal"]["SexxAge"].to_numpy()[yc == 1]).value_counts().to_dict()
                cal_ess = {c: kish(w["s_cal"][0][(yc == 1) & (g["s_cal"]["SexxAge"].to_numpy() == c)]) for c in cal_n}
                common = dict(study=study, seed=seed, imputation=j, outcome=o, model=mname, model_backend=MODEL_BACKEND.get(mname),
                              variant=variant, heldout=unit, n_s_cal=len(yc), n_cal_pos=int(yc.sum()),
                              ess_frac_cal=kish(w["s_cal"][0]) / len(yc), clip_value=clipv, **gib_info)
                tev = P["t_eval"][ok["t_eval"]]
                wdes = Ft["wdesign"].to_numpy()[tev]
                rr = np.random.default_rng(seed * 10007 + oi + unit_index * 31 + 101 * mi)
                for eo in ev_list:
                    ye = Ft[eo].to_numpy()[tev]
                    okk = ~np.isnan(ye); ye = ye[okk].astype(int); ge = g["t_eval"][okk].reset_index(drop=True); we = wdes[okk]
                    auc_t = roc_auc_score(ye, pr["t_eval"][okk]) if 0 < ye.mean() < 1 else np.nan
                    ref_m, ref_p = exact_reference(ye, ge, cal_n, ALPHA, CFG3["REF_SIMS"], rr)
                    refe_m, refe_p = exact_reference(ye, ge, {c: math.floor(v) for c, v in cal_ess.items()}, ALPHA, CFG3["REF_SIMS"], rr)
                    extra_masks = {}
                    if "undiagnosed" in Ft: extra_masks = {"undiag": Ft["undiagnosed"].to_numpy()[tev][okk] == 1,
                                                           "diag": Ft["diagnosed"].to_numpy()[tev][okk] == 1}
                    for meth, S in SETS.items():
                        Se = S[okk]
                        for wt in (["unweighted"] + (["design"] if not np.allclose(we, 1.0) else [])):
                            ww = None if wt == "unweighted" else we
                            m = metrics4(Se, ye, ge, w=ww)
                            for nm, mk in extra_masks.items():   # NHANES: share of the undiagnosed / diagnosed whose set contains 1
                                wv = np.ones(len(ye)) if ww is None else ww
                                m[f"cov1_{nm}"] = _wmean(Se[mk, 1].astype(float), wv[mk]); m[f"only1_{nm}"] = _wmean((Se[mk, 1] & ~Se[mk, 0]).astype(float), wv[mk])
                                m[f"zero_{nm}"] = _wmean((Se[mk, 0] & ~Se[mk, 1]).astype(float), wv[mk])
                            rows.append(dict(**common, eval_outcome=eo, eval_unit="ALL", method=meth, weighting=wt, auc_target=auc_t, **m,
                                             ref_exact_mean=ref_m, ref_exact_p05=ref_p, ref_ess_mean=refe_m, ref_ess_p05=refe_p))
                    if cutoffs:
                        cr = cutoff_rules(pc, yc, pr["t_eval"][okk], ye, we)
                        for nm, v in cr.items(): crow.append(dict(**common, eval_outcome=eo, rule=nm, **v))
                    if boot and j is None and (mname == "LR" or extra_masks):
                        stv, psv = Ft["strata"].to_numpy()[tev][okk], Ft["psu"].to_numpy()[tev][okk]
                        Wm = rw_multipliers(stv, psv, PSU_BY_STRATUM, CFG3["BOOT_DESIGN"], CFG3["BOOT_DESIGN_SEED"]) * we[None, :]
                        for meth in [m_ for m_ in ["ClassCond", "GroupCond", "SAGE-CP", "Weighted+ClassCond", "Gibbs-tuned", "ClassCond@0.20", "GroupCond@0.20"] if m_ in SETS]:
                            rm = rep_metrics(SETS[meth][okk], ye, ge, Wm)
                            for nm, mk in extra_masks.items():   # NHANES: share of the undiagnosed / diagnosed whose set contains 1, or {0}
                                Sm = SETS[meth][okk]
                                rm[f"cov1_{nm}"] = (Wm[:, mk] @ Sm[mk, 1].astype(float)) / Wm[:, mk].sum(1)
                                rm[f"zero_{nm}"] = (Wm[:, mk] @ (Sm[mk, 0] & ~Sm[mk, 1]).astype(float)) / Wm[:, mk].sum(1)
                            for met, arr in rm.items():
                                                brows.append(dict(study=study, seed=seed, heldout=unit, outcome=o, eval_outcome=eo, method=meth, metric=met, model=mname,
                                                  **{f"b{b}": float(v) for b, v in enumerate(arr)}))
    tag = f"{study}__{str(unit).replace(' ', '_').replace('>', '')}__s{seed}"
    _mark_done(key, tag, rows, {"gibbs": glog, "boot": brows, "cutoff": crow})
    print(f"{study} {unit} seed {seed}: {len(rows)} rows, {time.time() - t0:.0f}s", flush=True)
    return "ok"

def run_within_nhanes(seed, F, X):
    """N2: the whole pipeline inside NHANES with MEASURED total diabetes (50% train / 25% calibration / 25% evaluation)."""
    key = ("NHANES2", "within", int(seed))
    if key in DONE: return "done"
    rng = np.random.default_rng(90001 + seed); idx = rng.permutation(len(F)); n = len(F)
    tr, ca, ev = idx[: n // 2], idx[n // 2: 3 * n // 4], idx[3 * n // 4:]
    enc = make_encoder(HARM_NUM, HARM_CAT).fit(X.iloc[tr])
    rows, crow, brows = [], [], []
    for o in ["total", "total_fpg"]:
        y_all = F[o].to_numpy(); okt, okc, oke = (~np.isnan(y_all[s]) for s in (tr, ca, ev))
        if okc.sum() < 50: continue
        for mname in CFG3["MODELS"]["NHANES"]:
            model = make_model(mname, seed).fit(enc.transform(X.iloc[tr[okt]]), y_all[tr[okt]].astype(int))
            pc = model.predict_proba(enc.transform(X.iloc[ca[okc]]))[:, 1]; yc = y_all[ca[okc]].astype(int)
            pe = model.predict_proba(enc.transform(X.iloc[ev[oke]]))[:, 1]; ye = y_all[ev[oke]].astype(int)
            gc, ge = gframe(F, ca[okc]), gframe(F, ev[oke])
            kc = [gc["SexxAge"].to_numpy(str), gc["Age"].to_numpy(str), np.array(["all"] * len(gc))]
            ke = [ge["SexxAge"].to_numpy(str), ge["Age"].to_numpy(str), np.array(["all"] * len(ge))]
            SETS = build_sets(pc, yc, kc, pe, ke, None, None)
            wv = F["wfast" if o == "total_fpg" else "wdesign"].to_numpy()[ev[oke]]
            und = F["undiag_fpg" if o == "total_fpg" else "undiagnosed"].to_numpy()[ev[oke]] == 1
            cal_n = pd.Series(gc["SexxAge"].to_numpy()[yc == 1]).value_counts().to_dict()
            ref_m, ref_p = exact_reference(ye, ge, cal_n, ALPHA, CFG3["REF_SIMS"], rng)
            for meth, S in SETS.items():
                for wt, ww in [("unweighted", None), ("design", wv)]:
                    m = metrics4(S, ye, ge, w=ww); w1 = np.ones(len(ye)) if ww is None else ww
                    m.update(cov1_undiag=_wmean(S[und, 1].astype(float), w1[und]), only1_undiag=_wmean((S[und, 1] & ~S[und, 0]).astype(float), w1[und]),
                             zero_undiag=_wmean((S[und, 0] & ~S[und, 1]).astype(float), w1[und]))
                    rows.append(dict(study="NHANES2", seed=seed, outcome=o, eval_outcome=o, model=mname, method=meth, weighting=wt,
                                     auc_target=roc_auc_score(ye, pe), ref_exact_mean=ref_m, ref_exact_p05=ref_p, **m))
            if CFG3.get("NHANES_BOOT"):
                Wm = rw_multipliers(F["strata"].to_numpy()[ev[oke]], F["psu"].to_numpy()[ev[oke]], PSU_BY_STRATUM, CFG3["BOOT_DESIGN"], CFG3["BOOT_DESIGN_SEED"]) * wv[None, :]
                for meth in ["ClassCond", "GroupCond", "GroupCond@0.20"]:
                    S = SETS[meth]; rm = rep_metrics(S, ye, ge, Wm)
                    rm["cov1_undiag"] = (Wm[:, und] @ S[und, 1].astype(float)) / Wm[:, und].sum(1)
                    rm["zero_undiag"] = (Wm[:, und] @ (S[und, 0] & ~S[und, 1]).astype(float)) / Wm[:, und].sum(1)
                    for met, arr in rm.items():
                        brows.append(dict(study="NHANES2", seed=seed, heldout="within", outcome=o, eval_outcome=o, method=meth, metric=met, model=mname,
                                          **{f"b{b}": float(v) for b, v in enumerate(arr)}))
            for nm, v in cutoff_rules(pc, yc, pe, ye, wv).items():
                crow.append(dict(study="NHANES2", seed=seed, outcome=o, eval_outcome=o, model=mname, rule=nm, **v))
    _mark_done(key, f"NHANES2__within__s{seed}", rows, {"cutoff": crow, "boot": brows})
    print(f"NHANES2 seed {seed}: {len(rows)} rows", flush=True)
    return "ok"

def _loglik(y, p):
    p = np.clip(p, 1e-12, 1 - 1e-12); return float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))

def run_west(seed, F, X, num, cat):
    """Concept-shift diagnostics per held-out region and outcome (Kaggle extract, v2.1 full feature set).
    prev_target: observed prevalence in t_eval; prev_wsource: importance-weighted source prevalence on s_test (what a
    pure covariate shift with correct weights would predict for the target); oe: observed / mean predicted risk in the
    target; lr_p: likelihood-ratio test (2 df) of a region intercept and region x logit(p) slope in a logistic model of
    Y on logit(p) fitted to s_test + t_eval, i.e. whether P(Y | model risk) differs between source and target."""
    key = ("WEST", "all", int(seed))
    if key in DONE: return "done"
    rows = []
    reg = F["state"].map(REGION).to_numpy()
    for ui, r in enumerate(REGIONS_ALL):
        src, tgt = np.where(reg != r)[0], np.where(reg == r)[0]
        rng = np.random.default_rng(seed * 1_000_003 + ui * 7919 + 17 * STUDY_IX["WEST"])
        src = rng.permutation(src); tgt = rng.permutation(tgt)
        n_cal, n_st, n_dom = (min(CFG3[k], len(src) // 5) for k in ["N_CAL", "N_SRC_TEST", "N_DOM"])
        s_test = src[n_cal:n_cal + n_st]; s_dom = src[n_cal + n_st:n_cal + n_st + n_dom]
        s_train = _cap(src[n_cal + n_st + n_dom:], CFG3["MAX_TRAIN"], rng)
        t_unlab, t_eval = tgt[: len(tgt) // 2], tgt[len(tgt) // 2:]
        enc_b = make_encoder(num, cat).fit(X.iloc[s_train]); enc_d = make_encoder(num, cat).fit(pd.concat([X.iloc[s_dom], X.iloc[t_unlab]]))
        dm = fit_domain(enc_d.transform(X.iloc[s_dom]), enc_d.transform(X.iloc[t_unlab]))
        wr = weights_from(dm, enc_d.transform(X.iloc[s_test])); wcl = np.minimum(wr, np.quantile(weights_from(dm, enc_d.transform(X.iloc[s_dom])), CFG3["CLIP_Q"]))
        for o in CFG3["OUTCOMES"]:
            ys, yt, ytr = F[o].to_numpy()[s_test], F[o].to_numpy()[t_eval], F[o].to_numpy()[s_train]
            for mname in CFG3["MODELS"]["WEST"]:
                model = make_model(mname, seed).fit(enc_b.transform(X.iloc[s_train]), ytr.astype(int))
                ps = model.predict_proba(enc_b.transform(X.iloc[s_test]))[:, 1]; pt = model.predict_proba(enc_b.transform(X.iloc[t_eval]))[:, 1]
                z = np.r_[np.log(ps / (1 - ps)), np.log(pt / (1 - pt))]; d = np.r_[np.zeros(len(ps)), np.ones(len(pt))]; yy = np.r_[ys, yt].astype(int)
                from sklearn.linear_model import LogisticRegression as _LR
                m0 = _LR(penalty=None, max_iter=1000).fit(z[:, None], yy); m1 = _LR(penalty=None, max_iter=1000).fit(np.c_[z, d, z * d], yy)
                lr = 2 * (_loglik(yy, m1.predict_proba(np.c_[z, d, z * d])[:, 1]) - _loglik(yy, m0.predict_proba(z[:, None])[:, 1]))
                from scipy.stats import chi2
                dec = np.quantile(ps, np.linspace(0, 1, 11)); bs, bt = np.digitize(ps, dec[1:-1]), np.digitize(pt, dec[1:-1])
                gaps = [abs(yt[bt == k].mean() - ys[bs == k].mean()) for k in range(10) if (bt == k).sum() >= 50]
                rows.append(dict(study="WEST", seed=seed, region=r, outcome=o, model=mname, n_target=len(yt),
                                 prev_target=float(yt.mean()), prev_source=float(ys.mean()),
                                 prev_wsource_clipped=float(np.average(ys, weights=wcl)), prev_wsource_unclipped=float(np.average(ys, weights=wr)),
                                 mean_pred_target=float(pt.mean()), oe=float(yt.mean() / pt.mean()),
                                 auc_source=float(roc_auc_score(ys, ps)), auc_target=float(roc_auc_score(yt, pt)),
                                 lr_stat=float(lr), lr_p=float(chi2.sf(lr, 2)), region_intercept=float(m1.coef_[0][1]), region_slope=float(m1.coef_[0][2]),
                                 max_decile_gap=float(max(gaps)) if gaps else np.nan, mean_decile_gap=float(np.mean(gaps)) if gaps else np.nan))
    _mark_done(key, f"WEST__all__s{seed}", rows)
    print(f"WEST seed {seed}: {len(rows)} rows", flush=True)
    return "ok"


## 5. Run

In [ ]:
# ======================= 5. Run the v4 studies (resumable; each unit saved as soon as it finishes) =======================
def time_left(): return (time.time() - T0) / 60 < CFG3["TIME_BUDGET_MIN"]
DATA_INFO = {}
_prev_info = json.load(open(f"{OUT}/data_info.json")) if os.path.exists(f"{OUT}/data_info.json") else {}
def subset(X, num, cat, full=False):
    keep = list(X.columns) if full else [c for c in X.columns if c in CFG3["FEATURES_NO_PROXY"]]
    return X[keep], [c for c in num if c in keep], [c for c in cat if c in keep]
def units_of(F):
    vc = F["state"].value_counts(); return sorted(vc[vc >= CFG3["MIN_STATE_ROWS"]].index)
R22 = KF = None
def get_raw():
    global R22
    if R22 is None:
        R22 = load_raw(2022); DATA_INFO["raw2022"] = R22[4]
        F = R22[0]
        for h, d in F.groupby("strata")["psu"]: PSU_BY_STRATUM[h] = np.sort(d.unique())
    return R22
def get_kaggle():
    global KF
    if KF is None: KF = load_kaggle(); DATA_INFO["kaggle"] = KF[4]
    return KF
RAWP_VARIANT = {"RAWP": ("median", False), "RAWP_MISIND": ("misind", False), "RAWP_MI": ("mi", False), "RAWP_FULL": ("median", True), "RAWP_CB": ("median", False)}

for study in CFG3["STUDIES"]:
    if not time_left(): print("time budget reached: re-run the notebook to continue"); break
    try:
        if study == "NHANES":
            NF, NX, ninfo = load_nhanes(); DATA_INFO["nhanes"] = ninfo
            for h, d in NF.groupby("strata")["psu"]: PSU_BY_STRATUM[h] = np.sort(d.unique())
            F, X, num, cat, _ = get_raw()
            HX = brfss_harmonised(F, X); okb = (HX["Race4"] != "__MISSING__").to_numpy()
            Fb = F[okb].reset_index(drop=True); HXb = HX[okb].reset_index(drop=True)
            for seed in CFG3["SEEDS"]["NHANES"]:
                if not time_left(): break
                run_unit4("NHANES1", "BRFSS->NHANES", seed, Fb, HXb, np.arange(len(Fb)), NF, NX, np.arange(len(NF)), HARM_NUM, HARM_CAT,
                          src_outcomes=["HadDiabetes"], eval_outcomes={"HadDiabetes": ["diagnosed", "total"]}, cutoffs=True,
                          boot=CFG3.get("NHANES_BOOT", False))
                run_within_nhanes(seed, NF, NX)
        elif study == "WEST":
            F, X, num, cat, _ = get_kaggle()
            for seed in CFG3["SEEDS"]["WEST"]:
                if time_left(): run_west(seed, F, X, num, cat)
        elif study == "LOSO_G":
            F, X, num, cat, _ = get_kaggle(); Xn, numn, catn = subset(X, num, cat)
            for seed in CFG3["SEEDS"][study]:
                for ui, s in enumerate(units_of(F)):
                    if not time_left(): break
                    run_unit4(study, s, seed, F, Xn, np.where(F["state"] != s)[0], F, Xn, np.where(F["state"] == s)[0], numn, catn,
                              unit_index=ui, gibbs=True)
        elif study.startswith("RAWP"):
            F, X, num, cat, _ = get_raw()
            variant, full = RAWP_VARIANT[study]
            Xs_, nums_, cats_ = subset(X, num, cat, full=full)
            for seed in CFG3["SEEDS"][study]:
                for ui, s in enumerate(units_of(F)):
                    if not time_left(): break
                    run_unit4(study, s, seed, F, Xs_, np.where(F["state"] != s)[0], F, Xs_, np.where(F["state"] == s)[0], nums_, cats_,
                              unit_index=ui, variant=variant, gibbs=(study == "RAWP"), boot=(study == "RAWP" and seed == 0),
                              cutoffs=(study == "RAWP"))
    except Exception as e:
        traceback.print_exc(); check(f"study {study} completed without error", False, repr(e))
if RAW_MAP_LOG: pd.DataFrame(RAW_MAP_LOG).drop_duplicates().to_csv(f"{OUT}/raw_variable_map.csv", index=False)
json.dump({**_prev_info, **DATA_INFO}, open(f"{OUT}/data_info.json", "w"), indent=2, default=str)
print("elapsed min", round((time.time() - T0) / 60, 1))


## 6. Analysis

In [ ]:
# ======================= 6. Analysis (pre-specified contrasts; design-based and unit-bootstrap uncertainty) =======================
def _cat(pattern):
    fs = sorted(glob.glob(f"{UNIT_DIR}/{pattern}"))
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()
_all = sorted(f for f in glob.glob(f"{UNIT_DIR}/*.csv") if os.path.basename(f).count(".") == 1)
U = pd.concat([pd.read_csv(f) for f in _all], ignore_index=True) if _all else pd.DataFrame()
BT, CT, GL = _cat("*.boot.csv"), _cat("*.cutoff.csv"), _cat("*.gibbs.csv")
for nm, d in [("units", U), ("boot_design_replicates", BT), ("cutoff_rules", CT), ("gibbs_selection", GL)]:
    if len(d): d.to_csv(f"{OUT}/{nm}.csv.gz" if nm == "boot_design_replicates" else f"{OUT}/{nm}.csv", index=False)
check("v4 results present", len(U) > 0)
CONTRASTS = [
    ("GroupCond - ClassCond", "GroupCond", "ClassCond", "wg_cal_pos", "+"), ("GroupCond - ClassCond", "GroupCond", "ClassCond", "set_size", "-"),
    ("GroupCond - ClassCond", "GroupCond", "ClassCond", "wg_race_pos", "+"), ("SAGE-CP - GroupCond", "SAGE-CP", "GroupCond", "absdev_cov_pos", "-"),
    ("SAGE-CP - GroupCond", "SAGE-CP", "GroupCond", "wg_cal_pos", "+"), ("SAGE-CP - GroupCond", "SAGE-CP", "GroupCond", "set_size", "-"),
    ("Weighted+ClassCond - ClassCond", "Weighted+ClassCond", "ClassCond", "absdev_cov_pos", "-"),
    ("Gibbs-tuned - GroupCond", "Gibbs-tuned", "GroupCond", "wg_cal_pos", "+"), ("Gibbs-tuned - GroupCond", "Gibbs-tuned", "GroupCond", "absdev_cov_pos", "-"),
    ("Gibbs-tuned - GroupCond", "Gibbs-tuned", "GroupCond", "set_size", "-"),
]
METRICS = ["coverage", "cov_pos", "absdev_cov_pos", "set_size", "both", "zero", "single1", "miss0", "npv0", "ppv_single1", "wg_cal_pos",
           "wg_race_pos", "n_elig_cal", "n_elig_race", "auc_target", "ref_exact_mean", "ref_exact_p05", "ref_ess_mean", "ref_ess_p05", "n", "n_pos"]
def holm(p):
    p = np.asarray(p, float); o = np.argsort(np.where(np.isnan(p), np.inf, p)); m = np.sum(~np.isnan(p)); adj = np.full(len(p), np.nan); run = 0
    for r, i in enumerate(o):
        if np.isnan(p[i]): continue
        run = max(run, min(1.0, (m - r) * p[i])); adj[i] = run
    return adj
if len(U):
    L = U[U.study.isin(["RAWP", "RAWP_MISIND", "RAWP_MI", "RAWP_FULL", "RAWP_CB", "LOSO_G"])].copy()
    L["unit"] = L.heldout.astype(str)
    # unit level: average over seeds, models and imputations (Rubin's point estimate for RAWP_MI)
    UL = L.groupby(["study", "weighting", "outcome", "unit", "method"], as_index=False)[[c for c in METRICS if c in L]].mean(numeric_only=True)
    UP = UL.groupby(["study", "weighting", "unit", "method"], as_index=False).mean(numeric_only=True).assign(outcome="pooled")
    UL = pd.concat([UL, UP], ignore_index=True); UL.to_csv(f"{OUT}/v4_unit_level.csv", index=False)
    rb = np.random.default_rng(CFG3["BOOT_SEED"]); res = []
    for (st, wt, oc), d in UL.groupby(["study", "weighting", "outcome"]):
        for name, A, B_, met, better in CONTRASTS:
            diff = (d[d.method == A].set_index("unit")[met] - d[d.method == B_].set_index("unit")[met]).dropna()
            if len(diff) < 2: continue
            n = len(diff); bs = rb.choice(diff.to_numpy(), size=(CFG3["BOOT_STATES"], n)).mean(1); nz = diff[diff != 0]
            res.append(dict(study=st, weighting=wt, outcome=oc, contrast=name, metric=met, better=better, n_units=n, mean_diff=diff.mean(),
                            median_diff=diff.median(), ci95_lo=np.quantile(bs, .025), ci95_hi=np.quantile(bs, .975),
                            n_A_better=int(((diff > 0) if better == "+" else (diff < 0)).sum()), n_ties=int((diff == 0).sum()),
                            p_wilcoxon=wilcoxon(diff).pvalue if len(nz) else np.nan,
                            p_sign=binomtest(int((nz > 0).sum()), len(nz), 0.5).pvalue if len(nz) else np.nan))
    C = pd.DataFrame(res, columns=["study", "weighting", "outcome", "contrast", "metric", "better", "n_units", "mean_diff", "median_diff", "ci95_lo", "ci95_hi",
                                   "n_A_better", "n_ties", "p_wilcoxon", "p_sign"]); C["p_wilcoxon_holm"] = np.nan; C["holm_family_size"] = 0
    for _, ix in C.groupby(["study", "weighting", C.outcome == "pooled"]).groups.items():
        C.loc[ix, "p_wilcoxon_holm"] = holm(C.loc[ix, "p_wilcoxon"]); C.loc[ix, "holm_family_size"] = int(C.loc[ix, "p_wilcoxon"].notna().sum())
    # design-based SE of every pooled RAWP contrast (Rao-Wu replicates, seed 0; units and outcomes averaged within replicate)
    if len(BT):
        bc = [c for c in BT.columns if c.startswith("b") and c[1:].isdigit()]
        BTN = BT[BT.study.isin(["NHANES1", "NHANES2"])]
        if len(BTN):   # NHANES: replicate means over seeds (and models), then contrasts between methods
            g = BTN.groupby(["study", "eval_outcome", "model", "method", "metric"])[bc].mean()
            g.to_csv(f"{OUT}/v4_nhanes_replicates.csv")
        BT = BT[BT.study == "RAWP"]
        dz = []
        for name, A, B_, met, better in CONTRASTS:
            a = BT[(BT.method == A) & (BT.metric == met)].set_index(["heldout", "outcome"])[bc]
            b = BT[(BT.method == B_) & (BT.metric == met)].set_index(["heldout", "outcome"])[bc]
            if not len(a) or not len(b): continue
            dd = (a - b).dropna()
            for oc, sub in [("pooled", dd)] + [(o, dd.xs(o, level="outcome")) for o in dd.index.get_level_values("outcome").unique()]:
                rep = sub.groupby(level=0).mean().mean(0) if oc == "pooled" else sub.mean(0)
                dz.append(dict(study="RAWP", outcome=oc, contrast=name, metric=met, design_se=float(rep.std(ddof=1)),
                               design_lo=float(np.quantile(rep, .025)), design_hi=float(np.quantile(rep, .975)), n_replicates=len(bc)))
        DZ = pd.DataFrame(dz); DZ.to_csv(f"{OUT}/v4_design_se.csv", index=False)
        if len(DZ): C = C.merge(DZ.assign(weighting="design"), on=["study", "weighting", "outcome", "contrast", "metric"], how="left")
    C.to_csv(f"{OUT}/v4_contrasts.csv", index=False)
    SM = UL.groupby(["study", "weighting", "outcome", "method"]).mean(numeric_only=True); SM.to_csv(f"{OUT}/v4_summary_by_method.csv")
    # 17-state style check for RAWP: does the GroupCond gain follow state size and the number of eligible groups?
    d = UL[(UL.study == "RAWP") & (UL.weighting == "design") & (UL.outcome == "pooled")]
    if len(d):
        a = d[d.method == "GroupCond"].set_index("unit"); b = d[d.method == "ClassCond"].set_index("unit")
        from scipy.stats import spearmanr
        dg = a.wg_cal_pos - b.wg_cal_pos
        pd.DataFrame([dict(n_units=len(dg), n_helped=int((dg > 0).sum()), rho_n=spearmanr(dg, a.n).statistic,
                           rho_n_elig=spearmanr(dg, a.n_elig_cal).statistic, rho_classcond_wg=spearmanr(dg, b.wg_cal_pos).statistic)]).to_csv(f"{OUT}/v4_gc_gain_vs_size.csv", index=False)
    # NHANES summaries (seeds and base models averaged)
    N = U[U.study.isin(["NHANES1", "NHANES2"])]
    if len(N):
        cols = [c for c in ["coverage", "cov_pos", "set_size", "both", "zero", "single1", "miss0", "npv0", "ppv_single1", "wg_cal_pos",
                            "cov1_undiag", "only1_undiag", "zero_undiag", "cov1_diag", "auc_target", "ref_exact_mean", "n", "n_pos"] if c in N]
        N.groupby(["study", "eval_outcome", "weighting", "method"])[cols].mean().to_csv(f"{OUT}/v4_nhanes_summary.csv")
        N.groupby(["study", "eval_outcome", "weighting", "model", "method"])[cols].mean().to_csv(f"{OUT}/v4_nhanes_by_model.csv")
    if len(CT):
        CT.groupby(["study", "eval_outcome", "rule"]).mean(numeric_only=True).to_csv(f"{OUT}/v4_cutoff_summary.csv")
    W = U[U.study == "WEST"]
    if len(W):
        W.groupby(["region", "outcome"]).agg(prev_target=("prev_target", "mean"), prev_source=("prev_source", "mean"),
              prev_wsource_clipped=("prev_wsource_clipped", "mean"), oe=("oe", "mean"), auc_source=("auc_source", "mean"),
              auc_target=("auc_target", "mean"), lr_stat=("lr_stat", "median"), share_lr_p_below_001=("lr_p", lambda x: float((x < 0.001).mean())),
              region_intercept=("region_intercept", "mean"), region_slope=("region_slope", "mean"),
              max_decile_gap=("max_decile_gap", "mean")).to_csv(f"{OUT}/v4_west_concept_shift.csv")
    with pd.option_context("display.width", 220, "display.max_columns", 40):
        print(C[C.outcome == "pooled"].round(4).to_string())
pd.DataFrame(CHECKS).to_csv(f"{OUT}/checks_v4.csv", index=False)
zp = f"{OUT}/../SAGE_CP_v4_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{OUT}/**/*", recursive=True):
        if os.path.isfile(f) and not f.endswith(".zip"): z.write(f, os.path.relpath(f, OUT))
print("results zipped to", os.path.abspath(zp), "| elapsed min", round((time.time() - T0) / 60, 1))
